In [ ]:
import importlib.metadata
import os
import random
import subprocess
import sys
from pathlib import Path

def _ensure_transformers(min_ver=(5, 0, 0)):
    need_install = False
    try:
        ver_str = importlib.metadata.version("transformers").split("+")[0].split("a")[0].split("b")[0].split("rc")[0]
        parts = tuple(int(x) for x in ver_str.split(".")[:3])
        if parts < min_ver:
            need_install = True
    except Exception:
        need_install = True
    if need_install:
        print("[SETUP] Installing transformers>=5.0.0...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.0.0"], check=True)

_ensure_transformers()

import numpy as np
import torch
import torchvision
import transformers

IN_COLAB = "google.colab" in sys.modules
CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else Path("./checkpoints")
ENSEMBLE_DIR = CHECKPOINT_DIR / "ensemble"
OUTPUT_DIR_S2 = Path("/content/stage2_results") if IN_COLAB else Path("./stage2_results")
ZIP_EXPORT_S2 = Path("/content/SE374_Stage2_Training_Results.zip") if IN_COLAB else Path("./SE374_Stage2_Training_Results.zip")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_S2.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]
CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=True)
    CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())

STAGE2_GIT_REF = os.environ.get("STAGE2_GIT_REF", "").strip()
if STAGE2_GIT_REF:
    if not (Path(CODE_ROOT) / ".git").exists():
        raise RuntimeError(f"STAGE2_GIT_REF={STAGE2_GIT_REF!r} requested, but {CODE_ROOT} is not a git repo")
    subprocess.run(["git", "-C", CODE_ROOT, "reset", "--hard", "HEAD"], capture_output=True, text=True, check=True)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", "+refs/heads/*:refs/remotes/origin/*", "--tags"], capture_output=True, text=True, check=False)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--depth", "1", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=False)
    _br = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"refs/remotes/origin/{STAGE2_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
    if _br.returncode == 0:
        subprocess.run(["git", "-C", CODE_ROOT, "checkout", "-B", STAGE2_GIT_REF, f"refs/remotes/origin/{STAGE2_GIT_REF}"], capture_output=True, text=True, check=True)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
    else:
        if (Path(CODE_ROOT) / ".git" / "shallow").exists():
            subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--unshallow", "origin"], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "checkout", "--force", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
elif (Path(CODE_ROOT) / ".git").exists() and IN_COLAB:
    _hb = subprocess.run(["git", "-C", CODE_ROOT, "symbolic-ref", "--short", "-q", "HEAD"], capture_output=True, text=True, check=False)
    if _hb.returncode == 0 and _hb.stdout.strip():
        subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)

for _k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    sys.modules.pop(_k, None)
if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

SEED = int(os.environ.get("STAGE2_SEED", "42"))
DRY_RUN = os.environ.get("STAGE2_DRY_RUN", "0").strip().lower() in ("1", "true", "yes")
NUM_WORKERS = int(os.environ.get("STAGE2_NUM_WORKERS", "4"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (
    torch.bfloat16
    if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False))
    else torch.float16
)
print(f"[READY] Code={CODE_ROOT} | Mode=ensemble_eval | DryRun={DRY_RUN} | Seed={SEED}", flush=True)
print(f"[READY] Compute={DEVICE} | AMP={AMP_DTYPE} | torch={torch.__version__} | tv={torchvision.__version__} | tf={transformers.__version__}", flush=True)


In [ ]:
import urllib.request
import zipfile
from torch.utils.data import DataLoader
from src.colab_pkg.downloader import GDRIVE_DATASET_ID, HF_DATASET_URL, resolve_split_dirs
from src.training.dataset_loader import BCSSStage2ValDataset, load_bcss_patient_splits

DATASET_DIR = Path("/content/BCSS-WSSS") if IN_COLAB else Path("./data/BCSS-WSSS")
DATASET_ZIP = Path("/content/BCSS-WSSS.zip") if IN_COLAB else Path("./data/BCSS-WSSS.zip")
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

_, VAL_IMG_DIR, VAL_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="val", min_images=1, min_patients=1)
_, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)
if VAL_IMG_DIR is None or TEST_IMG_DIR is None:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1_000_000:
        print(f"[DOWNLOAD] Fetching BCSS-WSSS from {HF_DATASET_URL}", flush=True)
        try:
            urllib.request.urlretrieve(HF_DATASET_URL, str(DATASET_ZIP))
        except Exception:
            subprocess.run(["pip", "install", "-q", "gdown"], check=False)
            subprocess.run(["gdown", "--id", GDRIVE_DATASET_ID, "-O", str(DATASET_ZIP)], check=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(str(DATASET_DIR))
    _, VAL_IMG_DIR, VAL_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="val", min_images=1, min_patients=1)
    _, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)

if VAL_IMG_DIR is None or VAL_MASK_DIR is None or TEST_IMG_DIR is None or TEST_MASK_DIR is None:
    raise FileNotFoundError(f"[DATASET ERROR] Missing val/test dirs under {DATASET_DIR}")

splits = load_bcss_patient_splits()
val_pids = splits["val"]
test_pids = splits["test"]
assert set(splits["train"]).isdisjoint(set(val_pids)) and set(test_pids).isdisjoint(set(splits["train"]) | set(val_pids)), "Patient leakage detected!"

sample_lim = 64 if DRY_RUN else None
val_ds = BCSSStage2ValDataset(VAL_IMG_DIR, VAL_MASK_DIR, patient_ids=val_pids, limit=sample_lim, strict_patient_filter=True, seed=SEED)
test_ds = BCSSStage2ValDataset(TEST_IMG_DIR, TEST_MASK_DIR, patient_ids=test_pids, limit=sample_lim, strict_patient_filter=True, seed=SEED)
nw_eval = min(NUM_WORKERS, os.cpu_count() or 2)
dl_kw = {"prefetch_factor": 2, "multiprocessing_context": "spawn"} if nw_eval > 0 else {}
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=nw_eval, pin_memory=(DEVICE.type == "cuda"), **dl_kw)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=nw_eval, pin_memory=(DEVICE.type == "cuda"), **dl_kw)
print(f"[DATASET] Val={len(val_ds)} ({len(val_pids)} pats) | Test={len(test_ds)} ({len(test_pids)} pats)", flush=True)


In [ ]:
from src.colab_pkg.downloader import HF_STAGE1_URL, HF_STAGE2_URL
from src.models.wsss_inference_methods import safe_torch_load
from src.stage2_v2.ensemble import (
    DEFAULT_ARM_FAMILIES,
    DEFAULT_ARM_NAMES,
    SHA256_AUTHOR_STAGE1,
    SHA256_AUTHOR_STAGE2,
    SHA256_STAGE1_BEST_CLS,
    SHA256_STAGE1_RETRAINED,
)
from src.training.eval_stage1_full import compute_file_sha256

def _find_ckpt(candidates, expected_sha=None, download_url=None, label="ckpt"):
    path = next((p for p in candidates if p.is_file()), None)
    if path is None and download_url:
        path = candidates[0]
        print(f"[DOWNLOAD] Fetching {label} -> {path}", flush=True)
        urllib.request.urlretrieve(download_url, str(path))
    if path is None or not path.is_file():
        raise FileNotFoundError(f"[STAGE2 ERROR] Missing {label}: checked {[str(c) for c in candidates]}")
    sha = compute_file_sha256(path)
    if expected_sha and sha != expected_sha:
        raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {label} ({path}): got {sha}, expected {expected_sha}")
    print(f"[CKPT OK] {label}: {path} (sha256={sha[:12]})", flush=True)
    return path

# Resolve 4 Stage-2 arm checkpoints from ENSEMBLE_DIR or CHECKPOINT_DIR
arm_ckpt_map = {}
search_dirs = [ENSEMBLE_DIR, CHECKPOINT_DIR]
for arm_name in DEFAULT_ARM_NAMES:
    short_arm = arm_name.split("_")[0]
    cands = []
    for d in search_dirs:
        cands.extend([d / f"{arm_name}.pth", d / f"{short_arm}.pth", d / f"{short_arm}_best.pth"])
    hit = next((c for c in cands if c.is_file()), None)
    if hit is not None:
        arm_ckpt_map[arm_name] = hit

if len(arm_ckpt_map) < 4:
    for pth in sorted(ENSEMBLE_DIR.glob("*.pth")):
        try:
            meta = safe_torch_load(pth, device=torch.device("cpu"))
            preset = str(meta.get("preset", "")).strip().lower() if isinstance(meta, dict) else ""
            if preset in DEFAULT_ARM_NAMES and preset not in arm_ckpt_map:
                arm_ckpt_map[preset] = pth
        except Exception:
            pass

missing_arms = [a for a in DEFAULT_ARM_NAMES if a not in arm_ckpt_map]
if missing_arms:
    raise FileNotFoundError(f"[STAGE2 ERROR] Missing Stage-2 arm checkpoints for: {missing_arms} in {ENSEMBLE_DIR}")

arm_specs = []
for arm_name in DEFAULT_ARM_NAMES:
    ckpt_p = arm_ckpt_map[arm_name]
    sha = compute_file_sha256(ckpt_p)
    print(f"[ARM OK] {arm_name}: {ckpt_p.name} (sha256={sha[:12]})", flush=True)
    arm_specs.append({"name": arm_name, "family": DEFAULT_ARM_FAMILIES[arm_name], "ckpt_path": ckpt_p})

STAGE1_BEST_CLS_PATH = _find_ckpt(
    [ENSEMBLE_DIR / "stage1_best_cls.pth", CHECKPOINT_DIR / "stage1_best_cls.pth"],
    expected_sha=SHA256_STAGE1_BEST_CLS,
    label="stage1_best_cls.pth",
)
STAGE1_RETRAINED_PATH = _find_ckpt(
    [ENSEMBLE_DIR / "stage1_retrained_best.pth", CHECKPOINT_DIR / "stage1_retrained_best.pth"],
    expected_sha=SHA256_STAGE1_RETRAINED,
    label="stage1_retrained_best.pth",
)
AUTHOR_STAGE1_PATH = _find_ckpt(
    [ENSEMBLE_DIR / "stage1_checkpoint_trained_on_bcss.pth", CHECKPOINT_DIR / "stage1_checkpoint_trained_on_bcss.pth"],
    expected_sha=SHA256_AUTHOR_STAGE1,
    download_url=HF_STAGE1_URL,
    label="stage1_checkpoint_trained_on_bcss.pth",
)
AUTHOR_STAGE2_PATH = _find_ckpt(
    [ENSEMBLE_DIR / "stage2_checkpoint_trained_on_bcss.pth", CHECKPOINT_DIR / "stage2_checkpoint_trained_on_bcss.pth"],
    expected_sha=SHA256_AUTHOR_STAGE2,
    download_url=HF_STAGE2_URL,
    label="stage2_checkpoint_trained_on_bcss.pth",
)


In [ ]:
from src.stage2_v2.ensemble import run_stage2_v2_ensemble_eval

metrics = run_stage2_v2_ensemble_eval(
    arm_specs=arm_specs,
    val_loader=val_loader,
    test_loader=test_loader,
    stage1_cls_ckpt=STAGE1_BEST_CLS_PATH,
    stage1_norm_mode="zero_one",
    author_stage2_ckpt=AUTHOR_STAGE2_PATH,
    author_stage1_retrained_ckpt=STAGE1_RETRAINED_PATH,
    author_stage1_official_ckpt=AUTHOR_STAGE1_PATH,
    output_dir=OUTPUT_DIR_S2,
    zip_path=ZIP_EXPORT_S2,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    n_bootstrap=10000,
    n_perm=10000,
    seed=SEED,
)
print("[ENSEMBLE DONE] Primary 4-Arm TEST mIoU:", metrics["test_primary_4arm"]["mIoU"], flush=True)


In [ ]:
sel = metrics["val_protocol_selection"]
print("=== FROZEN VAL PROTOCOL ===", flush=True)
print("Selected TTAs    :", sel["selected_ttas"], flush=True)
print("Cls Prob Source  :", sel["selected_cls_source"], flush=True)
print("Prob Mode/Weights:", sel["selected_prob_mode"], sel["selected_weight_strategy"], sel["primary_weights"], flush=True)
print("CGM Strategy     :", sel["cgm_expert_strategy"], sel["cgm_dof"], sel["cgm_thresholds"], "use_cgm=", sel["use_cgm"], flush=True)
print("Pre-Registered   : single=", sel["val_best_single_name"], "| pair=", sel["val_best_cnn_vit_pair_name"], flush=True)
print("=== TEST SUMMARY (23 DISJOINT PATIENTS) ===", flush=True)
for ak, av in metrics.get("test_author_anchors", {}).items():
    print(f"  Author {ak}: 1x_mul0={av['cgm_author_mul0']['mIoU']:.4f} | ms_tta={av['ms_tta']['mIoU']:.4f} | ms_tta_cgm={av['ms_tta_cgm_neg1e4']['mIoU']:.4f}", flush=True)
for k, v in metrics["test_individual_arms"].items():
    print(f"  Arm {k}: matched({v['cgm_mode']})={v['mIoU']:.4f} (no_cgm={v['no_cgm']['mIoU']:.4f}) 95%CI={v['bootstrap_95ci']}", flush=True)
vp = metrics["test_val_best_cnn_vit_pair"]
print(f"  Best CNN+ViT  : {vp['pair_name']} matched({vp['cgm_mode']})={vp['mIoU']:.4f} (no_cgm={vp['no_cgm']['mIoU']:.4f}) 95%CI={vp['bootstrap_95ci']}", flush=True)
print(f"  Equal 4-Arm   : mIoU={metrics['test_equal_4arm'][metrics['test_primary_4arm']['cgm_mode']]['mIoU']:.4f}", flush=True)
print(f"  Primary 4-Arm : mIoU={metrics['test_primary_4arm']['mIoU']:.4f} 95%CI={metrics['test_primary_4arm']['bootstrap_95ci']}", flush=True)
diag = metrics.get("test_pure_ensemble_diagnostic", {}).get("primary_4arm_no_cgm_vs_best_single_no_cgm")
if diag:
    print(f"  Pure-Ens Diag : 4arm_no_cgm={diag['primary_4arm_no_cgm_miou']:.4f} vs {diag['best_single_no_cgm_name']}={diag['best_single_no_cgm_miou']:.4f} (delta={diag['delta_miou']:+.4f}, p={diag['perm_p_raw']:.6f})", flush=True)
for row in metrics["test_confirmatory_stats_holm_m5"]:
    print("  Stat:", row, flush=True)


In [ ]:
metrics_json = OUTPUT_DIR_S2 / "STAGE2_TRAINING_METRICS.json"
plot_png = OUTPUT_DIR_S2 / "stage2_training_loss.png"
for req_f in (metrics_json, plot_png, ZIP_EXPORT_S2):
    if not req_f.is_file():
        raise FileNotFoundError(f"[EXPORT ERROR] Missing expected deliverable: {req_f}")
zip_kb = ZIP_EXPORT_S2.stat().st_size / 1024.0
print(f"[EXPORT OK] JSON={metrics_json} | PNG={plot_png} | ZIP={ZIP_EXPORT_S2} ({zip_kb:.1f} KB)", flush=True)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_S2))
    except Exception as exc:
        print(f"[NOTICE] Colab browser download skipped: {exc}", flush=True)
